# Regularization Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: Dropout (Train and Eval Mode)

In [ ]:
```python

import random

import math

class Dropout:

    def __init__(self, p=0.5):

        self.p = p

        self.training = True

        self.mask = None

    def forward(self, x):

        if not self.training:

            return list(x)

        self.mask = []

        output = []

        for val in x:

            if random.random() < self.p:

                self.mask.append(0)

                output.append(0.0)

            else:

                self.mask.append(1)

                output.append(val / (1 - self.p))

        return output

    def backward(self, grad_output):

        grads = []

        for g, m in zip(grad_output, self.mask):

            if m == 0:

                grads.append(0.0)

            else:

                grads.append(g / (1 - self.p))

        return grads

In [ ]:
```

### Step 2: L2 Weight Decay

In [ ]:
```python

def l2_regularization(weights, lambda_reg):

    penalty = 0.0

    for w in weights:

        penalty += w * w

    return lambda_reg * 0.5 * penalty

def l2_gradient(weights, lambda_reg):

    return [lambda_reg * w for w in weights]

In [ ]:
```

### Step 3: Batch Normalization

In [ ]:
```python

class BatchNorm:

    def __init__(self, num_features, momentum=0.1, eps=1e-5):

        self.gamma = [1.0] * num_features

        self.beta = [0.0] * num_features

        self.eps = eps

        self.momentum = momentum

        self.running_mean = [0.0] * num_features

        self.running_var = [1.0] * num_features

        self.training = True

        self.num_features = num_features

    def forward(self, batch):

        batch_size = len(batch)

        if self.training:

            mean = [0.0] * self.num_features

            for sample in batch:

                for j in range(self.num_features):

                    mean[j] += sample[j]

            mean = [m / batch_size for m in mean]

            var = [0.0] * self.num_features

            for sample in batch:

                for j in range(self.num_features):

                    var[j] += (sample[j] - mean[j]) ** 2

            var = [v / batch_size for v in var]

            for j in range(self.num_features):

                self.running_mean[j] = (1 - self.momentum) * self.running_mean[j] + self.momentum * mean[j]

                self.running_var[j] = (1 - self.momentum) * self.running_var[j] + self.momentum * var[j]

        else:

            mean = list(self.running_mean)

            var = list(self.running_var)

        self.x_hat = []

        output = []

        for sample in batch:

            normalized = []

            out_sample = []

            for j in range(self.num_features):

                x_h = (sample[j] - mean[j]) / math.sqrt(var[j] + self.eps)

                normalized.append(x_h)

                out_sample.append(self.gamma[j] * x_h + self.beta[j])

            self.x_hat.append(normalized)

            output.append(out_sample)

        return output

In [ ]:
```

### Step 4: Layer Normalization

In [ ]:
```python

class LayerNorm:

    def __init__(self, num_features, eps=1e-5):

        self.gamma = [1.0] * num_features

        self.beta = [0.0] * num_features

        self.eps = eps

        self.num_features = num_features

    def forward(self, x):

        mean = sum(x) / len(x)

        var = sum((xi - mean) ** 2 for xi in x) / len(x)

        self.x_hat = []

        output = []

        for j in range(self.num_features):

            x_h = (x[j] - mean) / math.sqrt(var + self.eps)

            self.x_hat.append(x_h)

            output.append(self.gamma[j] * x_h + self.beta[j])

        return output

In [ ]:
```

### Step 5: RMSNorm

In [ ]:
```python

class RMSNorm:

    def __init__(self, num_features, eps=1e-6):

        self.gamma = [1.0] * num_features

        self.eps = eps

        self.num_features = num_features

    def forward(self, x):

        rms = math.sqrt(sum(xi * xi for xi in x) / len(x) + self.eps)

        output = []

        for j in range(self.num_features):

            output.append(self.gamma[j] * x[j] / rms)

        return output

In [ ]:
```

### Step 6: Training With and Without Regularization

In [ ]:
```python

def sigmoid(x):

    x = max(-500, min(500, x))

    return 1.0 / (1.0 + math.exp(-x))

def make_circle_data(n=200, seed=42):

    random.seed(seed)

    data = []

    for _ in range(n):

        x = random.uniform(-2, 2)

        y = random.uniform(-2, 2)

        label = 1.0 if x * x + y * y < 1.5 else 0.0

        data.append(([x, y], label))

    return data

class RegularizedNetwork:

    def __init__(self, hidden_size=16, lr=0.05, dropout_p=0.0, weight_decay=0.0):

        random.seed(0)

        self.hidden_size = hidden_size

        self.lr = lr

        self.dropout_p = dropout_p

        self.weight_decay = weight_decay

        self.dropout = Dropout(p=dropout_p) if dropout_p > 0 else None

        self.w1 = [[random.gauss(0, 0.5) for _ in range(2)] for _ in range(hidden_size)]

        self.b1 = [0.0] * hidden_size

        self.w2 = [random.gauss(0, 0.5) for _ in range(hidden_size)]

        self.b2 = 0.0

    def forward(self, x, training=True):

        self.x = x

        self.z1 = []

        self.h = []

        for i in range(self.hidden_size):

            z = self.w1[i][0] * x[0] + self.w1[i][1] * x[1] + self.b1[i]

            self.z1.append(z)

            self.h.append(max(0.0, z))

        if self.dropout and training:

            self.dropout.training = True

            self.h = self.dropout.forward(self.h)

        elif self.dropout:

            self.dropout.training = False

            self.h = self.dropout.forward(self.h)

        self.z2 = sum(self.w2[i] * self.h[i] for i in range(self.hidden_size)) + self.b2

        self.out = sigmoid(self.z2)

        return self.out

    def backward(self, target):

        eps = 1e-15

        p = max(eps, min(1 - eps, self.out))

        d_loss = -(target / p) + (1 - target) / (1 - p)

        d_sigmoid = self.out * (1 - self.out)

        d_out = d_loss * d_sigmoid

        for i in range(self.hidden_size):

            d_relu = 1.0 if self.z1[i] > 0 else 0.0

            d_h = d_out * self.w2[i] * d_relu

            self.w2[i] -= self.lr * (d_out * self.h[i] + self.weight_decay * self.w2[i])

            for j in range(2):

                self.w1[i][j] -= self.lr * (d_h * self.x[j] + self.weight_decay * self.w1[i][j])

            self.b1[i] -= self.lr * d_h

        self.b2 -= self.lr * d_out

    def evaluate(self, data):

        correct = 0

        total_loss = 0.0

        for x, y in data:

            pred = self.forward(x, training=False)

            eps = 1e-15

            p = max(eps, min(1 - eps, pred))

            total_loss += -(y * math.log(p) + (1 - y) * math.log(1 - p))

            if (pred >= 0.5) == (y >= 0.5):

                correct += 1

        return total_loss / len(data), correct / len(data) * 100

    def train_model(self, train_data, test_data, epochs=300):

        history = []

        for epoch in range(epochs):

            total_loss = 0.0

            correct = 0

            for x, y in train_data:

                pred = self.forward(x, training=True)

                self.backward(y)

                eps = 1e-15

                p = max(eps, min(1 - eps, pred))

                total_loss += -(y * math.log(p) + (1 - y) * math.log(1 - p))

                if (pred >= 0.5) == (y >= 0.5):

                    correct += 1

            train_loss = total_loss / len(train_data)

            train_acc = correct / len(train_data) * 100

            test_loss, test_acc = self.evaluate(test_data)

            history.append((train_loss, train_acc, test_loss, test_acc))

            if epoch % 75 == 0 or epoch == epochs - 1:

                gap = train_acc - test_acc

                print(f"    Epoch {epoch:3d}: train_acc={train_acc:.1f}%, test_acc={test_acc:.1f}%, gap={gap:.1f}%")

        return history

In [ ]:
```

## Exercises

In [ ]:
1. Implement spatial dropout for 2D data: instead of dropping individual neurons, drop entire feature channels. Simulate this by treating groups of consecutive features as channels and dropping whole groups. Compare the train-test gap to standard dropout on the circle dataset with hidden_size=32.

2. Implement label smoothing from lesson 05 combined with dropout from this lesson. Train with four configurations: neither, dropout only, label smoothing only, both. Measure the final train-test accuracy gap for each. Which combination gives the smallest gap?

3. Add a BatchNorm layer between the hidden layer and the activation in your circle-dataset network. Train with and without BatchNorm at learning rates 0.01, 0.05, and 0.1. BatchNorm should allow stable training at higher learning rates where the vanilla network diverges.

4. Implement early stopping: track test loss each epoch, save the best weights, and stop if test loss hasn't improved for 20 epochs. Run the regularized network for 1000 epochs. Report which epoch had the best test accuracy and how many epochs of computation you saved.

5. Compare LayerNorm vs RMSNorm on a 4-layer network (not just 2). Initialize both with the same weights. Train for 200 epochs and compare final accuracy, training speed (time per epoch), and gradient magnitudes at the first layer. Verify that RMSNorm is faster with the same accuracy.